## Proof of concept - Profit Exposure
### QUESTION:
How can leadership monitor policy profitability and claims exposure over time?
### INTERPRETATION:
What is profit exposure? My assumption is again that income/profit is payments + endorsements - claims. I've set up the data set to track this quarterly per policy. This could be set up monthly, but the data seemed too sporadic for that.


In [ ]:
%%sql -r quarterly_customer
select * from rli.present.prs_quarterly_policy_data;

In [ ]:
import plotly.graph_objects as go
import pandas as pd

df = quarterly_customer.copy()
df['NET_INCOME'] = df['NET_INCOME'].astype(float)
df['CLAIMS'] = df['CLAIMS'].astype(float)
df['PAYMENTS_ENDORSEMENTS'] = df['PAYMENTS_ENDORSEMENTS'].astype(float)

customers = sorted(df['CUSTOMER_NAME'].unique().tolist())

all_quarterly = df.groupby('YEARQUARTER').agg(
    net_income=('NET_INCOME', 'sum'),
    claims=('CLAIMS', 'sum'),
    payments_endorsements=('PAYMENTS_ENDORSEMENTS', 'sum')
).reset_index().sort_values('YEARQUARTER')

fig = go.Figure()

fig.add_trace(go.Bar(x=all_quarterly['YEARQUARTER'], y=all_quarterly['payments_endorsements'], name='Payments & Endorsements', marker_color='steelblue', opacity=0.7, visible=True))
fig.add_trace(go.Bar(x=all_quarterly['YEARQUARTER'], y=all_quarterly['claims'], name='Claims', marker_color='salmon', opacity=0.7, visible=True))
fig.add_trace(go.Scatter(x=all_quarterly['YEARQUARTER'], y=all_quarterly['net_income'], name='Net Income', line=dict(color='green', width=2), mode='lines+markers', visible=True))

buttons = [dict(label='All Customers', method='update', args=[{'x': [all_quarterly['YEARQUARTER'], all_quarterly['YEARQUARTER'], all_quarterly['YEARQUARTER']], 'y': [all_quarterly['payments_endorsements'], all_quarterly['claims'], all_quarterly['net_income']]}, {'title': 'Quarterly Profit Exposure: All Customers'}])]

for cust in customers:
    cust_df = df[df['CUSTOMER_NAME'] == cust].groupby('YEARQUARTER').agg(
        net_income=('NET_INCOME', 'sum'),
        claims=('CLAIMS', 'sum'),
        payments_endorsements=('PAYMENTS_ENDORSEMENTS', 'sum')
    ).reset_index().sort_values('YEARQUARTER')
    buttons.append(dict(label=cust, method='update', args=[{'x': [cust_df['YEARQUARTER'], cust_df['YEARQUARTER'], cust_df['YEARQUARTER']], 'y': [cust_df['payments_endorsements'], cust_df['claims'], cust_df['net_income']]}, {'title': f'Quarterly Profit Exposure: {cust}'}]))

fig.update_layout(
    title='Quarterly Profit Exposure: All Customers',
    xaxis_title='Quarter',
    yaxis_title='Amount ($)',
    updatemenus=[dict(buttons=buttons, direction='down', showactive=True, x=0.0, xanchor='left', y=1.15, yanchor='top')],
    barmode='group',
    height=600
)

fig.show()